# Prior Rollout Evaluation from Neutral Pose

This notebook demonstrates:
1. Loading a trained track-mjx checkpoint (PPO or Distilled)
2. Running prior-only rollouts starting from the neutral pose
3. Termination checking during rollouts
4. Unit testing: verifying deterministic rollouts produce identical results

All necessary functions are defined inline in this notebook.

In [1]:
# Add packages to path if not installed
import sys
sys.path.insert(0, '/home/mila/a/aidan.sirbu/track-mjx')
sys.path.insert(0, '/home/mila/a/aidan.sirbu/vnl-playground')

In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys
import collections
import logging
from typing import Callable, Dict, Tuple, Any, Sequence, Optional, Union

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

# Set rendering backend
if sys.platform == "darwin":
    os.environ["MUJOCO_GL"] = "glfw"
    print("Using macOS with GLFW rendering")
else:
    os.environ["MUJOCO_GL"] = "egl"
    print("Using Linux with EGL rendering")

import jax
import jax.numpy as jnp
from jax import random
import numpy as np
from ml_collections import config_dict
from omegaconf import OmegaConf, DictConfig
import imageio
import mediapy as media
import matplotlib.pyplot as plt
from mujoco import mjx
from mujoco_playground._src import mjx_env
import flax
from flax import linen as nn
from brax.training import distribution, networks, types
from brax.training.types import PRNGKey
from brax.training.acme import running_statistics, specs
from orbax import checkpoint as ocp

# vnl-playground imports for environment
from vnl_playground.tasks.rodent import imitation
from vnl_playground.tasks.rodent.wrappers import FlattenObsWrapper

print(f"JAX version: {jax.__version__}")
print(f"Available devices: {jax.devices()}")

Using Linux with EGL rendering


JAX version: 0.7.2
Available devices: [CudaDevice(id=0)]


## Configuration

In [17]:
# Path to your track-mjx checkpoint
CHECKPOINT_PATH = "/home/mila/a/aidan.sirbu/scratch/track-mjx/model_checkpoints/251213_235327_736369"

# Checkpoint type: "PPONetwork" for PPO checkpoints, "DistillNetwork" for distilled models
STEP_PREFIX = "DistillNetwork"

# Rollout settings
NUM_STEPS = 200          # Number of steps per rollout
NUM_ROLLOUTS = 4         # Number of rollouts to generate
FIXED_LOGVAR = -2.0      # Fixed log-variance for prior sampling
DETERMINISTIC = False     # If True, use prior mean (no sampling)
NEUTRAL_Z = 0          # Z-height for neutral pose (within healthy range)

# Termination settings
HEALTHY_Z_RANGE = (0.0325, 0.5)  # Healthy torso height range

## Network Definitions

Define the Prior, Decoder, and IntentionNetwork classes needed for loading checkpoints.

In [18]:
def reparameterize(rng: PRNGKey, mean: jnp.ndarray, logvar: jnp.ndarray) -> jnp.ndarray:
    """Reparameterization trick for VAE sampling."""
    std = jnp.exp(0.5 * logvar)
    eps = random.normal(rng, logvar.shape)
    return mean + eps * std


class Encoder(nn.Module):
    """Encoder network that outputs distributions in latent space."""
    layer_sizes: Sequence[int]
    latents: int
    activation: networks.ActivationFn = nn.silu
    kernel_init: networks.Initializer = jax.nn.initializers.lecun_uniform()
    bias: bool = True

    @nn.compact
    def __call__(self, x: jnp.ndarray, get_activation: bool = False):
        activations = {}
        for i, hidden_size in enumerate(self.layer_sizes):
            x = nn.Dense(hidden_size, name=f"hidden_{i}", kernel_init=self.kernel_init, use_bias=self.bias)(x)
            x = self.activation(x)
            x = nn.LayerNorm()(x)
            if get_activation:
                activations[f"layer_{i}"] = x
        mean_x = nn.Dense(self.latents, name="fc2_mean")(x)
        logvar_x = nn.Dense(self.latents, name="fc2_logvar")(x)
        if get_activation:
            return (mean_x, logvar_x), activations
        return mean_x, logvar_x


class Decoder(nn.Module):
    """Decoder network that outputs actions from latent and proprioceptive observations."""
    layer_sizes: Sequence[int]
    activation: networks.ActivationFn = nn.silu
    kernel_init: networks.Initializer = jax.nn.initializers.lecun_uniform()
    activate_final: bool = False
    bias: bool = True

    @nn.compact
    def __call__(self, x: jnp.ndarray, get_activation: bool = False):
        activations = {}
        for i, hidden_size in enumerate(self.layer_sizes):
            x = nn.Dense(hidden_size, name=f"hidden_{i}", kernel_init=self.kernel_init, use_bias=self.bias)(x)
            if i != len(self.layer_sizes) - 1 or self.activate_final:
                x = self.activation(x)
                x = nn.LayerNorm()(x)
                if get_activation:
                    activations[f"layer_{i}"] = x
        if get_activation:
            return x, activations
        return x, {}


class Prior(nn.Module):
    """Prior network that outputs distributions in latent space from proprioceptive observations."""
    layer_sizes: Sequence[int]
    latents: int
    activation: networks.ActivationFn = nn.silu
    kernel_init: networks.Initializer = jax.nn.initializers.lecun_uniform()
    bias: bool = True

    @nn.compact
    def __call__(self, x: jnp.ndarray, get_activation: bool = False):
        activations = {}
        for i, hidden_size in enumerate(self.layer_sizes):
            x = nn.Dense(hidden_size, name=f"hidden_{i}", kernel_init=self.kernel_init, use_bias=self.bias)(x)
            x = self.activation(x)
            x = nn.LayerNorm()(x)
            if get_activation:
                activations[f"layer_{i}"] = x
        mean_x = nn.Dense(self.latents, name="fc2_mean")(x)
        logvar_x = nn.Dense(self.latents, name="fc2_logvar")(x)
        if get_activation:
            return (mean_x, logvar_x), activations
        return mean_x, logvar_x


class IntentionNetwork(nn.Module):
    """Full VAE-based intention network with prior, encoder, and decoder."""
    encoder_layers: Sequence[int]
    decoder_layers: Sequence[int]
    prior_layers: Sequence[int]
    reference_obs_size: int
    latents: int = 60

    def setup(self):
        self.encoder = Encoder(layer_sizes=self.encoder_layers, latents=self.latents)
        self.decoder = Decoder(layer_sizes=self.decoder_layers)
        self.prior = Prior(layer_sizes=self.prior_layers, latents=self.latents)

    def __call__(self, obs: jnp.ndarray, key: PRNGKey, deterministic: bool = False, get_activation: bool = False):
        _, encoder_rng = jax.random.split(key)
        traj = obs[..., : self.reference_obs_size]
        egocentric_obs = obs[..., self.reference_obs_size :]

        encoder_input = jnp.concatenate([traj, egocentric_obs], axis=-1)
        latent_mean, latent_logvar = self.encoder(encoder_input, get_activation=False)
        prior_mean, prior_logvar = self.prior(egocentric_obs, get_activation=False)

        if deterministic:
            z = latent_mean
        else:
            z = reparameterize(encoder_rng, latent_mean, latent_logvar)

        action, _ = self.decoder(jnp.concatenate([z, egocentric_obs], axis=-1))
        return action, latent_mean, latent_logvar, prior_mean, prior_logvar


print("Network classes defined!")

Network classes defined!


## Checkpoint Loading Functions

In [19]:
def make_intention_policy(
    action_param_size: int,
    latent_size: int,
    total_obs_size: int,
    reference_obs_size: int,
    preprocess_observations_fn = types.identity_observation_preprocessor,
    encoder_hidden_layer_sizes: Sequence[int] = (1024, 1024),
    decoder_hidden_layer_sizes: Sequence[int] = (1024, 1024),
    prior_hidden_layer_sizes: Sequence[int] = (1024, 1024),
) -> networks.FeedForwardNetwork:
    """Create a policy network with intention module."""
    policy_module = IntentionNetwork(
        encoder_layers=list(encoder_hidden_layer_sizes),
        decoder_layers=list(decoder_hidden_layer_sizes) + [action_param_size],
        prior_layers=list(prior_hidden_layer_sizes),
        reference_obs_size=reference_obs_size,
        latents=latent_size,
    )

    def apply(processor_params, policy_params, obs, key, deterministic: bool = False, get_activation: bool = False):
        obs = preprocess_observations_fn(obs, processor_params)
        return policy_module.apply(policy_params, obs=obs, key=key, deterministic=deterministic, get_activation=get_activation)

    dummy_total_obs = jnp.zeros((1, total_obs_size))
    dummy_key = jax.random.PRNGKey(0)

    return networks.FeedForwardNetwork(
        init=lambda key: policy_module.init(key, dummy_total_obs, dummy_key),
        apply=apply,
    )


def make_abstract_policy_from_cfg(cfg: Union[Dict, DictConfig], seed: int = 1) -> tuple:
    """Create an abstract policy to define the pytree structure for loading."""
    network_config = cfg.get("network_config", cfg)
    
    # Check if normalize_observations is enabled
    normalize = lambda x, y: x
    train_config = cfg.get("train_setup", {}).get("train_config", {})
    if train_config.get("normalize_observations", False):
        normalize = running_statistics.normalize
    
    # Get layer sizes
    prior_layer_sizes = network_config.get("prior_layer_sizes", network_config.get("encoder_layer_sizes"))
    
    # Create action distribution to get param size
    action_size = network_config["action_size"]
    parametric_action_distribution = distribution.NormalTanhDistribution(event_size=action_size)
    
    # Create policy network
    policy_network = make_intention_policy(
        action_param_size=parametric_action_distribution.param_size,
        latent_size=network_config["intention_size"],
        total_obs_size=network_config["observation_size"],
        reference_obs_size=network_config["reference_obs_size"],
        preprocess_observations_fn=normalize,
        encoder_hidden_layer_sizes=tuple(network_config["encoder_layer_sizes"]),
        decoder_hidden_layer_sizes=tuple(network_config["decoder_layer_sizes"]),
        prior_hidden_layer_sizes=tuple(prior_layer_sizes),
    )
    
    key_policy = jax.random.key(seed)
    policy_params = policy_network.init(key_policy)
    
    # Create abstract normalizer state
    normalizer_state = running_statistics.init_state(
        specs.Array(network_config["observation_size"], jnp.dtype("float32"))
    )
    
    return (normalizer_state, policy_params)


def load_checkpoint_for_eval(
    checkpoint_path: str,
    step_prefix: str = "PPONetwork",
    step: Optional[int] = None,
) -> Dict[str, Any]:
    """Load a checkpoint's config and policy for evaluation."""
    mgr_options = ocp.CheckpointManagerOptions(create=False, step_prefix=step_prefix)
    ckpt_mgr = ocp.CheckpointManager(checkpoint_path, options=mgr_options)
    
    if step is None:
        step = ckpt_mgr.latest_step()
    print(f"Loading checkpoint from {checkpoint_path} at step {step}")

    # Load config first
    cfg = OmegaConf.create(
        ckpt_mgr.restore(
            step,
            args=ocp.args.Composite(config=ocp.args.JsonRestore()),
        )["config"]
    )

    # Create abstract policy to define pytree structure
    abstract_policy = make_abstract_policy_from_cfg(cfg)
    
    # Load policy
    policy = ckpt_mgr.restore(
        step,
        args=ocp.args.Composite(policy=ocp.args.StandardRestore(abstract_policy)),
    )["policy"]

    return {"cfg": cfg, "policy": policy}


print("Checkpoint loading functions defined!")

Checkpoint loading functions defined!


## Prior Rollout Functions

In [20]:
def check_termination(
    data: Any,
    healthy_z_range: Tuple[float, float] = (0.0325, 0.5),
) -> jax.Array:
    """Check if a state should be terminated."""
    from jax import flatten_util
    flattened_vals, _ = flatten_util.ravel_pytree(data.qpos)
    flattened_qvel, _ = flatten_util.ravel_pytree(data.qvel)
    all_vals = jnp.concatenate([flattened_vals, flattened_qvel])
    has_nan = jnp.any(jnp.isnan(all_vals))
    torso_z = data.qpos[2]
    min_z, max_z = healthy_z_range
    z_out_of_range = jnp.logical_or(torso_z < min_z, torso_z > max_z)
    return jnp.logical_or(has_nan, z_out_of_range)


def create_neutral_state(env, mjx_model, neutral_z: float = 0.1) -> mjx_env.State:
    """Create an environment state initialized to the neutral pose."""
    neutral_qpos = jnp.array(env.mj_model.qpos0)
    neutral_qpos = neutral_qpos.at[2].set(neutral_z)
    
    data = mjx.make_data(mjx_model)
    data = data.replace(qpos=neutral_qpos)
    data = data.replace(qvel=jnp.zeros(mjx_model.nv))
    data = mjx.forward(mjx_model, data)
    
    info = {
        "prev_action": jnp.zeros(env.action_size),
        "action": jnp.zeros(env.action_size),
        "start_frame": 0,
        "reference_clip": 0,
    }
    
    proprioception = env._get_proprioception(data, info, flatten=False)
    obs = collections.OrderedDict(proprioception=proprioception)
    
    return mjx_env.State(data, obs, jnp.array(0.0), jnp.array(0.0), {}, info)


def create_prior_policy(
    prior_network_params: Dict,
    decoder_network_params: Dict,
    normalizer_params: running_statistics.RunningStatisticsState,
    intention_latent_size: int,
    action_size: int,
    proprioceptive_obs_size: int,
    decoder_hidden_layer_sizes: Sequence[int],
    prior_hidden_layer_sizes: Sequence[int],
    fixed_logvar: float = -2.0,
    deterministic: bool = False,
) -> Callable:
    """Create a policy function that uses only the prior and decoder networks."""
    parametric_action_distribution = distribution.NormalTanhDistribution(event_size=action_size)
    
    prior_module = Prior(layer_sizes=list(prior_hidden_layer_sizes), latents=intention_latent_size)
    decoder_module = Decoder(layer_sizes=list(decoder_hidden_layer_sizes) + [parametric_action_distribution.param_size])
    
    def policy_fn(obs: jax.Array, rng_key: jax.Array) -> Tuple[jax.Array, Dict[str, Any]]:
        key_sample, key_action = random.split(rng_key, 2)

        proprioceptive_obs = obs[-proprioceptive_obs_size:]

        normalized_obs = running_statistics.normalize(proprioceptive_obs, normalizer_params)
        prior_mean, prior_logvar = prior_module.apply({"params": prior_network_params}, normalized_obs)
        
        fixed_logvar_array = jnp.full_like(prior_mean, fixed_logvar)
        
        if deterministic:
            z = prior_mean
        else:
            z = reparameterize(key_sample, prior_mean, fixed_logvar_array)
        
        decoder_input = jnp.concatenate([z, normalized_obs], axis=-1)
        logits, _ = decoder_module.apply({"params": decoder_network_params}, decoder_input)
        
        if deterministic:
            action = parametric_action_distribution.mode(logits)
        else:
            raw_action = parametric_action_distribution.sample_no_postprocessing(logits, key_action)
            action = parametric_action_distribution.postprocess(raw_action)
        
        extras = {
            "prior_mean": prior_mean,
            "prior_logvar": prior_logvar,
            "intention": z,
            "logits": logits,
        }
        return action, extras
    
    return policy_fn


def run_prior_rollout(
    env,
    policy_fn: Callable,
    jit_step: Callable,
    jit_reset: Callable,
    jit_create_neutral_state: Callable,
    rng_key: jax.Array,
    num_steps: int,
    healthy_z_range: Tuple[float, float] = (0.0325, 0.5),
    clip_idx: int=0,
    start_frame: int=0,
) -> Dict:
    """Run a single prior rollout starting from neutral pose."""
    from jax import flatten_util
    
    # state = jit_create_neutral_state()
    state = jit_reset(rng_key,
                    clip_idx=clip_idx,
                    start_frame=start_frame,)
    rollout_states = [state]
    actions = []
    prior_means = []
    intentions = []
    termination_step = None
    
    for step_idx in range(num_steps):
        rng_key, action_key = random.split(rng_key)
        
        if hasattr(state.obs, 'get') or isinstance(state.obs, dict):
            proprio = state.obs.get("proprioception", state.obs)
            if isinstance(proprio, dict):
                proprio, _ = flatten_util.ravel_pytree(proprio)
        else:
            proprio = state.obs
        
        action, extras = policy_fn(proprio, action_key)
        actions.append(action)
        prior_means.append(extras["prior_mean"])
        intentions.append(extras["intention"])
        
        state = jit_step(state, action)
        rollout_states.append(state)
        
        is_terminated = check_termination(state.data, healthy_z_range)
        if is_terminated and termination_step is None:
            termination_step = step_idx + 1
            print(f"TERMINATED at step {termination_step}")
            print(f"  Torso z: {state.data.qpos[2]:.4f}")
    
    return {
        "rollout_states": rollout_states,
        "actions": actions,
        "prior_means": prior_means,
        "intentions": intentions,
        "terminated": termination_step is not None,
        "termination_step": termination_step,
        "final_z": float(state.data.qpos[2]),
    }


print("Prior rollout functions defined!")

Prior rollout functions defined!


## 1. Load Checkpoint

In [21]:
# Load checkpoint
ckpt = load_checkpoint_for_eval(CHECKPOINT_PATH, step_prefix=STEP_PREFIX, step=None)
cfg = ckpt["cfg"]
full_policy_params = ckpt["policy"]

# Extract config values
network_config = cfg.get("network_config", cfg)
prior_layer_sizes = network_config.get("prior_layer_sizes", network_config.get("encoder_layer_sizes"))

proprioceptive_obs_size = network_config["proprioceptive_obs_size"]
action_size = network_config["action_size"]
intention_size = network_config["intention_size"]
decoder_layer_sizes = tuple(network_config["decoder_layer_sizes"])
prior_layer_sizes = tuple(prior_layer_sizes)

print(f"Checkpoint loaded!")
print(f"Proprioceptive obs size: {proprioceptive_obs_size}")
print(f"Action size: {action_size}")
print(f"Intention latent size: {intention_size}")
print(f"Decoder layers: {decoder_layer_sizes}")
print(f"Prior layers: {prior_layer_sizes}")

Loading checkpoint from /home/mila/a/aidan.sirbu/scratch/track-mjx/model_checkpoints/251213_235327_736369 at step 87
Checkpoint loaded!
Proprioceptive obs size: 264
Action size: 38
Intention latent size: 60
Decoder layers: (512, 512, 256, 256)
Prior layers: (512, 256, 256)


In [22]:
# Extract prior and decoder parameters
normalizer_params = full_policy_params[0]
full_network_params = full_policy_params[1]

# Extract only proprioceptive portion of normalizer
proprio_normalizer_params = running_statistics.RunningStatisticsState(
    count=normalizer_params.count,
    mean=normalizer_params.mean[-proprioceptive_obs_size:],
    summed_variance=normalizer_params.summed_variance[-proprioceptive_obs_size:],
    std=normalizer_params.std[-proprioceptive_obs_size:],
)

prior_params = full_network_params["params"]["prior"]
decoder_params = full_network_params["params"]["decoder"]

print("Prior and decoder parameters extracted!")

Prior and decoder parameters extracted!


## 2. Create Environment

In [23]:
def create_environment(cfg):
    """Create the imitation environment from checkpoint config."""
    env_cfg = cfg.get("env_config", cfg)
    if hasattr(env_cfg, 'to_container'):
        env_cfg_dict = OmegaConf.to_container(env_cfg, resolve=True)
    else:
        env_cfg_dict = dict(env_cfg)
    env_cfg_ml = config_dict.ConfigDict(env_cfg_dict)
    return FlattenObsWrapper(imitation.Imitation(config=env_cfg_ml))

env = create_environment(cfg)

# print(f"Environment created!")
# print(f"Observation size: {env.observation_size}")
# print(f"Proprioceptive obs size: {env.proprioceptive_obs_size}")
# print(f"Action size: {env.action_size}")

## 3. Create Prior Policy

In [24]:
prior_policy = create_prior_policy(
    prior_network_params=prior_params,
    decoder_network_params=decoder_params,
    normalizer_params=proprio_normalizer_params,
    intention_latent_size=intention_size,
    action_size=action_size,
    proprioceptive_obs_size=proprioceptive_obs_size,
    decoder_hidden_layer_sizes=decoder_layer_sizes,
    prior_hidden_layer_sizes=prior_layer_sizes,
    fixed_logvar=FIXED_LOGVAR,
    deterministic=DETERMINISTIC,
)

jit_prior_policy = jax.jit(prior_policy)
jit_step = jax.jit(env.step)
jit_reset = jax.jit(env.reset)
jit_create_neutral_state = jax.jit(lambda: create_neutral_state(env, env.mjx_model, neutral_z=NEUTRAL_Z))

print(f"Prior policy created!")
print(f"Deterministic: {DETERMINISTIC}")
print(f"Fixed logvar: {FIXED_LOGVAR}")

Prior policy created!
Deterministic: False
Fixed logvar: -2.0


## 4. Visualize Neutral Pose

In [11]:
neutral_state = jit_create_neutral_state()

print(f"Neutral state created!")
print(f"Neutral qpos (first 10): {neutral_state.data.qpos[:10]}")
print(f"Neutral z-height: {neutral_state.data.qpos[2]}")
print(f"Healthy z range: {HEALTHY_Z_RANGE}")

is_terminated = check_termination(neutral_state.data, HEALTHY_Z_RANGE)
print(f"Initial termination check: {'TERMINATED' if is_terminated else 'OK'}")

KeyboardInterrupt: 

## 5. Run Prior Rollouts

In [25]:
rollouts = []
base_key = random.PRNGKey(42)

print(f"Running {NUM_ROLLOUTS} prior rollouts from neutral pose...")
print(f"Deterministic: {DETERMINISTIC}")
print(f"Num steps: {NUM_STEPS}")
print("-" * 50)

for i in range(NUM_ROLLOUTS):
    print(f"\nRollout {i + 1}:")
    key = random.fold_in(base_key, i)
    
    rollout = run_prior_rollout(
        env=env,
        policy_fn=jit_prior_policy,
        jit_step=jit_step,
        jit_reset=jit_reset,
        jit_create_neutral_state=jit_create_neutral_state,
        rng_key=key,
        num_steps=NUM_STEPS,
        healthy_z_range=HEALTHY_Z_RANGE,
        clip_idx=4,
        start_frame=0,
    )
    rollouts.append(rollout)
    
    if not rollout["terminated"]:
        print(f"  Completed {NUM_STEPS} steps without termination")
    print(f"  Final z: {rollout['final_z']:.4f}")

print("\n" + "=" * 50)
print("Summary:")
for i, rollout in enumerate(rollouts):
    status = f"terminated at step {rollout['termination_step']}" if rollout['terminated'] else "completed"
    print(f"  Rollout {i + 1}: {status}, final z = {rollout['final_z']:.4f}")

Running 4 prior rollouts from neutral pose...
Deterministic: False
Num steps: 200
--------------------------------------------------

Rollout 1:
TERMINATED at step 1
  Torso z: 0.0087
  Final z: 0.1254

Rollout 2:
TERMINATED at step 1
  Torso z: 0.0125
  Final z: -0.0160

Rollout 3:
TERMINATED at step 1
  Torso z: 0.0117
  Final z: -0.0143

Rollout 4:
TERMINATED at step 1
  Torso z: 0.0076
  Final z: 0.0366

Summary:
  Rollout 1: terminated at step 1, final z = 0.1254
  Rollout 2: terminated at step 1, final z = -0.0160
  Rollout 3: terminated at step 1, final z = -0.0143
  Rollout 4: terminated at step 1, final z = 0.0366


## 6. Unit Test: Verify Deterministic Rollouts are Identical

In [44]:
def test_deterministic_rollouts(rollouts: list, deterministic: bool) -> bool:
    """Test that deterministic rollouts produce identical results."""
    if not deterministic:
        print("Skipping test: deterministic=False, rollouts expected to differ")
        return True
    
    if len(rollouts) < 2:
        print("Need at least 2 rollouts to compare")
        return True
    
    print("Testing deterministic rollouts are identical...")
    print("-" * 50)
    
    all_passed = True
    reference = rollouts[0]
    
    for i, rollout in enumerate(rollouts[1:], start=2):
        print(f"\nComparing rollout 1 vs rollout {i}:")
        
        # Compare actions
        actions_match = True
        for step_idx, (a1, a2) in enumerate(zip(reference["actions"], rollout["actions"])):
            if not jnp.allclose(a1, a2, atol=1e-6):
                actions_match = False
                print(f"  FAIL: Actions differ at step {step_idx}")
                break
        if actions_match:
            print(f"  PASS: All actions identical")
        else:
            all_passed = False
        
        # Compare prior means
        prior_means_match = True
        for step_idx, (m1, m2) in enumerate(zip(reference["prior_means"], rollout["prior_means"])):
            if not jnp.allclose(m1, m2, atol=1e-6):
                prior_means_match = False
                print(f"  FAIL: Prior means differ at step {step_idx}")
                break
        if prior_means_match:
            print(f"  PASS: All prior means identical")
        else:
            all_passed = False
        
        # Compare final states
        final_qpos_1 = reference["rollout_states"][-1].data.qpos
        final_qpos_2 = rollout["rollout_states"][-1].data.qpos
        if jnp.allclose(final_qpos_1, final_qpos_2, atol=1e-5):
            print(f"  PASS: Final qpos identical")
        else:
            print(f"  FAIL: Final qpos differ")
            all_passed = False
        
        # Compare termination
        if reference["termination_step"] == rollout["termination_step"]:
            print(f"  PASS: Termination step identical ({reference['termination_step']})")
        else:
            print(f"  FAIL: Termination step differs")
            all_passed = False
    
    print("\n" + "=" * 50)
    if all_passed:
        print("ALL TESTS PASSED: Deterministic rollouts are identical!")
    else:
        print("TESTS FAILED: Deterministic rollouts differ!")
    
    return all_passed

test_passed = test_deterministic_rollouts(rollouts, DETERMINISTIC)

Testing deterministic rollouts are identical...
--------------------------------------------------

Comparing rollout 1 vs rollout 2:
  FAIL: Actions differ at step 23
  FAIL: Prior means differ at step 30
  FAIL: Final qpos differ
  PASS: Termination step identical (25)

Comparing rollout 1 vs rollout 3:
  FAIL: Actions differ at step 17
  FAIL: Prior means differ at step 22
  FAIL: Final qpos differ
  PASS: Termination step identical (25)

Comparing rollout 1 vs rollout 4:
  FAIL: Actions differ at step 17
  FAIL: Prior means differ at step 41
  FAIL: Final qpos differ
  PASS: Termination step identical (25)

TESTS FAILED: Deterministic rollouts differ!


## 7. Render Rollout Video

In [28]:
import mujoco

def render_prior_rollout(env, rollout_states, height=480, width=640, camera="side-rodent"):
    """Render a prior rollout."""
    renderer = mujoco.Renderer(env.mj_model, height=height, width=width)
    mj_data = mujoco.MjData(env.mj_model)
    
    frames = []
    for state in rollout_states:
        mj_data.qpos[:] = np.array(state.data.qpos)
        mj_data.qvel[:] = np.array(state.data.qvel)
        mujoco.mj_forward(env.mj_model, mj_data)
        renderer.update_scene(mj_data, camera=camera)
        frames.append(renderer.render())
    return frames

print("Rendering first rollout...")
video_frames = render_prior_rollout(env, rollouts[3]["rollout_states"])
print(f"Rendered {len(video_frames)} frames")

media.show_video(video_frames, fps=50)

Rendering first rollout...
Rendered 201 frames


This browser does not support the video tag.


In [13]:
rollouts[0]['rollout_states'][0].__dict__.keys()

dict_keys(['data', 'obs', 'reward', 'done', 'metrics', 'info'])

In [14]:
rollouts[0]['rollout_states'][0].obs

Array([-4.26378608e-01, -2.55113006e-01,  8.22223499e-02, -3.50820959e-01,
       -3.12788755e-01, -6.83341101e-02,  4.15724963e-01, -4.56502438e-01,
       -7.24357739e-02,  1.17352251e-02,  3.99499714e-01,  0.00000000e+00,
       -2.44699165e-01,  3.07322085e-01, -3.20231408e-01, -6.38699710e-01,
       -9.36895460e-02,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  1.35207832e-01,  4.55748707e-01,
       -1.52485713e-01,  1.35701180e-01,  2.15477675e-01, -6.97437748e-02,
        6.10865235e-01, -8.72664601e-02,  0.00000000e+00,  1.24199636e-01,
       -2.61775434e-01, -

## 8. Plot Z-Height

In [ ]:
z_heights = [state.data.qpos[2] for state in rollouts[0]["rollout_states"]]

plt.figure(figsize=(12, 4))
plt.plot(z_heights, label='Torso Z')
plt.axhline(y=HEALTHY_Z_RANGE[0], color='r', linestyle='--', label=f'Min healthy ({HEALTHY_Z_RANGE[0]})')
plt.axhline(y=HEALTHY_Z_RANGE[1], color='r', linestyle='--', label=f'Max healthy ({HEALTHY_Z_RANGE[1]})')
plt.axhline(y=NEUTRAL_Z, color='g', linestyle=':', label=f'Neutral z ({NEUTRAL_Z})')
if rollouts[0]["terminated"]:
    plt.axvline(x=rollouts[0]["termination_step"], color='orange', linestyle='--', label='Termination')
plt.xlabel("Step")
plt.ylabel("Z Height")
plt.title("Torso Z-Height Over Time")
plt.legend()
plt.tight_layout()
plt.show()